In [7]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset, random_split
import torchvision.datasets as datasets
import os
%matplotlib inline

In [8]:
def load_data(Jd, l, num_conf, T, num_temps, batch_size, shuffle_opt, opt='train'):
    datasets = []
    for j in range(num_temps):
        path = f'data_spins/{Jd}_{opt}/spins_{l}_{T[j]}.npy'
        #if os.path.isfile(path):
        with open(path, 'rb') as f:
            x = np.load(f)   
        tensor_x = torch.Tensor(x).unsqueeze(1)
        path = f'data_spins/{Jd}_{opt}/answ_{l}_{T[j]}.npy'
        #if os.path.isfile(path):
        with open(path, 'rb') as f:
            y = np.load(f)
        tensor_y = torch.from_numpy(y).type(torch.float32)

        datasets.append(TensorDataset(tensor_x, tensor_y))


    dataset = torch.utils.data.ConcatDataset(datasets)

    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle_opt)

In [18]:
class Net(nn.Module):
    def __init__(self, l):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 64, 2)
        self.pool = nn.MaxPool2d(2, 2)
        self.act_hid = nn.ReLU()
        self.fc1 = nn.Linear(64*int(l/2-1)*int(l/2-1), 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.conv1(x)
        x = self.pool(x)
        x = self.act_hid(x)
        x = x.view(-1, 64*int(l/2-1)*int(l/2-1))
        x = self.fc1(x)
        x = self.act_hid(x)
        x = self.fc2(x)
        return x

In [29]:
def train(l, train_dataloader, num_epoch, criterion, batch_size):
    model = Net(l)
    optimizer = torch.optim.Adam(model.parameters(), lr=1.0e-4)
    act = nn.Sigmoid()

    for epoch in range(num_epoch):  
        running_loss = 0.0
        accuracy = 0.0
        pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader))
        for i, data in pbar:
            inputs, labels = data

            optimizer.zero_grad()
            outputs = model(inputs)
            #outputs = act(outputs)

            outputs = outputs.squeeze(1) # к одной размерности с labels

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            accuracy += (batch_size - sum(abs(labels - act(outputs)))).float().mean()

            pbar.set_description(
                    f"Loss: {running_loss/((i+1)*batch_size)} "
                    f"Accuracy: {accuracy * 100  / ((i+1)*batch_size)}"
            )

    print('Training completed')
    return model

In [11]:
def testing(model, test_dataloader, criterion, batch_size):
    outp = []
    errors = []
    accuracy = 0.0
    act = nn.Sigmoid()
    with torch.no_grad():
        for i, data in enumerate(test_dataloader):
            inputs, labels = data
            outputs = model(inputs)
            #outputs = act(outputs)
            outputs = outputs.squeeze(1)
            outp.append(act(outputs).item())
            loss = criterion(outputs, labels)
            errors.append(loss.item())

            accuracy += (1 - sum(abs(labels - act(outputs)))).float().mean()

    print("Accuracy = {}".format(accuracy / len(test_dataloader)))
    return outp, errors

In [12]:
roots = [2.2691853142129728, 2.104982167992544, 1.932307699120554, 1.749339162933206, 1.5536238493280832, 1.34187327905057, 1.109960313758399, 0.8541630993606272, 0.5762735442012712, 0.2885386111960936, 0.03198372863548067]
jds = [0.0, -0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1.0]
get_crit_T = dict(zip(jds, roots))

In [30]:
### training ###

def train_and_save(Jd, l, num_temps):
    num_conf_tr = 2048
    num_conf_ts = 512
    num_epoch = 1

    T_c = get_crit_T[Jd]
    T = np.round(np.linspace(T_c - 0.3, T_c + 0.3, num_temps), 4)

    criterion = nn.BCEWithLogitsLoss()     

    train_dataloader = load_data(Jd, l, num_conf_tr, T, num_temps, batch_size=4, shuffle_opt=True, opt='train')
    print(f'Start training for L = {l}')
    model = train(l, train_dataloader, num_epoch, criterion, batch_size=4)

    PATH = f'models/{l}_{Jd}_{T[0]}_{T[-1]}_{num_temps}.pt'
    torch.save(model.state_dict(), PATH)

In [36]:
L = 80
Jd = 0.0
num_temps = 100
T_c = get_crit_T[Jd]
T = np.round(np.linspace(T_c - 0.3, T_c + 0.3, num_temps), 4)
print(T)
train_and_save(Jd, L, num_temps)

[1.9692 1.9752 1.9813 1.9874 1.9934 1.9995 2.0055 2.0116 2.0177 2.0237
 2.0298 2.0359 2.0419 2.048  2.054  2.0601 2.0662 2.0722 2.0783 2.0843
 2.0904 2.0965 2.1025 2.1086 2.1146 2.1207 2.1268 2.1328 2.1389 2.1449
 2.151  2.1571 2.1631 2.1692 2.1752 2.1813 2.1874 2.1934 2.1995 2.2055
 2.2116 2.2177 2.2237 2.2298 2.2359 2.2419 2.248  2.254  2.2601 2.2662
 2.2722 2.2783 2.2843 2.2904 2.2965 2.3025 2.3086 2.3146 2.3207 2.3268
 2.3328 2.3389 2.3449 2.351  2.3571 2.3631 2.3692 2.3752 2.3813 2.3874
 2.3934 2.3995 2.4055 2.4116 2.4177 2.4237 2.4298 2.4359 2.4419 2.448
 2.454  2.4601 2.4662 2.4722 2.4783 2.4843 2.4904 2.4965 2.5025 2.5086
 2.5146 2.5207 2.5268 2.5328 2.5389 2.5449 2.551  2.5571 2.5631 2.5692]


FileNotFoundError: [Errno 2] No such file or directory: 'data_spins/0.0_train/spins_80_1.9995.npy'

In [24]:
### testing ###

def get_errs_outs(Jd, l, num_temps):
    T_c = get_crit_T[Jd]
    T = np.linspace(T_c - 0.3, T_c + 0.3, num_temps)
    T = np.round(T, 4)

    num_conf_tr = 2048
    num_conf_ts = 512

    criterion = nn.BCEWithLogitsLoss()     
    
    model = Net(l)
    T_c_ = get_crit_T[0.0]
    T_ = np.round(np.linspace(T_c_ - 0.3, T_c_ + 0.3, num_temps), 4)
    PATH = f'models/{l}_0.0_{T_[0]}_{T_[-1]}_{num_temps}.pt'

    model.load_state_dict(torch.load(PATH))
    model.eval()
    test_dataloader = load_data(Jd, l, num_conf_ts, T, num_temps, batch_size=1, shuffle_opt=False, opt='test')
    outp, errors = testing(model, test_dataloader, criterion, batch_size=1)
    return errors, outp

In [34]:
L = [30, 60]
Jd = 0.0
num_temps = 100
for l in L:
    errs = get_errs_outs(Jd, l, num_temps)[0]
    np.save(f'data_errors/{Jd}_{l}_{num_temps}.npy', errs)

Accuracy = 0.8926475644111633


FileNotFoundError: [Errno 2] No such file or directory: 'data_spins/0.0_test/spins_60_1.9813.npy'

In [49]:
def get_errs_outs_fr(Jd, L_):
    num_temps = 100
    T_c = get_crit_T[Jd]
    T = np.linspace(T_c - 0.3, T_c + 0.3, num_temps)
    T = np.round(T, 4)

    num_conf_tr = 2048
    num_conf_ts = 512

    criterion = nn.BCEWithLogitsLoss()     

    for i in range(len(L_)):
        L = L_[i]
        model = Net()
        T_c_ = get_crit_T[0.0]
        T_ = np.round(np.linspace(T_c_ - 0.3, T_c_ + 0.3, num_temps), 4)
        PATH = f'models/{L}_0.0_{T_[0]}_{T_[-1]}_{num_temps}.pt'
        model.load_state_dict(torch.load(PATH))
        model.eval()
        test_dataloader = load_data(Jd, L, num_conf_ts, T, num_temps, batch_size=1, shuffle_opt=False, opt='test')
        outp_i, errors_i = testing(model, test_dataloader, criterion, batch_size=1)
        np.save(f'data_errors/{Jd}_{L}_{num_temps}.npy', errors_i)

Accuracy = 0.901121199131012
